# Visualising the QQQ momentum backtest

Run the cells top to bottom. The notebook walks through the three stages the program
goes through, in order:

1. **Stock selection** — which names survive the trend / base / liquidity screen each
   month, and how they rank on relative strength.
2. **Portfolio construction** — the target book the ranking turns into: weights, cash
   residual, composition drift and turnover.
3. **Profit and loss** — equity curve against QQQ buy & hold, drawdowns, monthly
   returns and trade-level P&L.

Everything is computed by the repo's own modules (`config.py`, `datasource.py`,
`signals.py`, `backtest.py`) — this notebook only draws what they produce, so what you
see here is exactly what `python main.py` runs.

**Before you start**

```
pip install -r requirements.txt
```

Price history comes from whichever backend `config.toml` names. `yfinance` needs
nothing; `ib` needs TWS or IB Gateway running and logged in with the API enabled
(see the README). The first run downloads ~100 symbols and caches them under
`./data_cache/<source>/`, so re-runs are fast.

## 0. Setup

In [ ]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

# The strategy modules live in the repo root. Walk up from wherever Jupyter was
# started until we find them, then work from there -- the data cache and the
# CSV/PNG outputs are all written relative to the repo root.
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "backtest.py").exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "backtest.py").exists():
    raise RuntimeError("Could not find backtest.py -- run this notebook from the repo")
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)

from config import Config
from datasource import create_data_source
from qqq_universe import get_qqq_universe
import backtest as bt
import signals as sig

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)
plt.rcParams.update({
    "figure.figsize": (11, 4.5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})

# One palette for the whole notebook, so the same thing is the same colour everywhere.
STRAT, BENCH, CASH_C = "#2b6cb0", "#8d99ae", "#adb5bd"
POS, NEG, ACCENT = "#2f9e44", "#c92a2a", "#e8590c"

print(f"working in {REPO_ROOT}")

## 1. Configuration

Everything below is driven by `config.toml`. Change the two knobs in this cell to
override the data source or force a re-download for this session only; edit
`config.toml` for anything else.

In [ ]:
DATA_SOURCE = None    # None = use config.toml; or "yfinance" / "ib" for this run only
REFRESH = False       # True re-downloads instead of reading ./data_cache

cfg = Config.load()

settings = pd.Series({
    "data source": DATA_SOURCE or cfg.data_source,
    "measured window": f"{cfg.backtest_years} years (+{cfg.warmup_years}y warmup)",
    "initial capital": f"${cfg.initial_capital:,.0f}",
    "max positions": cfg.max_positions,
    "position sizing": cfg.position_sizing,
    "rebalance": "first trading day of each month",
    "RS benchmark": cfg.benchmark_ticker,
    "compared against": f"{cfg.compare_ticker} buy & hold",
    "trend template": f"close > SMA{cfg.sma_mid} > SMA{cfg.sma_long}, "
                      f"SMA{cfg.sma_short} > SMA{cfg.sma_mid}",
    "52w high/low band": f"within {cfg.pct_below_52w_high_max:.0f}% of high, "
                         f"{cfg.pct_above_52w_low_min:.0f}%+ above low",
    "base tightness": f"{cfg.base_lookback_weeks}w range <= {cfg.base_max_range_pct:.0f}%",
    "exit rule": f"close below SMA{cfg.sma_long}, checked daily",
    "transaction cost": f"{cfg.txn_cost_bps:.1f} bps per leg",
}, name="value")

display(settings.to_frame())

## 2. Load price history

Cached bars are read straight off disk. The **first** run has to download the whole
universe — a minute or so on `yfinance`, a few minutes on `ib` (which is pacing-limited),
and with `ib` the gateway has to be running or this cell will raise.

In [ ]:
universe = get_qqq_universe()
source = create_data_source(cfg, override=DATA_SOURCE)
print(f"data source: {source.name}   universe: {len(universe)} symbols")

with source:
    bench_df = source.fetch_daily_bars(cfg.benchmark_ticker, use_cache=not REFRESH)
    compare_df = source.fetch_daily_bars(cfg.compare_ticker, use_cache=not REFRESH)
    raw_data = source.fetch_universe_bars(universe, use_cache=not REFRESH)

missing = sorted(set(universe) - set(raw_data))
print(f"\nloaded {len(raw_data)}/{len(universe)} symbols"
      + (f"; no data for: {', '.join(missing)}" if missing else ""))

coverage = pd.DataFrame({
    "first bar": {s: df.index.min().date() for s, df in raw_data.items()},
    "last bar": {s: df.index.max().date() for s, df in raw_data.items()},
    "bars": {s: len(df) for s, df in raw_data.items()},
})
print(f"benchmark {cfg.benchmark_ticker}: {bench_df.index.min().date()} -> "
      f"{bench_df.index.max().date()} ({len(bench_df)} bars)")
display(coverage.sort_values("bars").head(5))   # the shortest histories, worth a look

## 3. Run the backtest

`run_backtest` steps day by day: the exit rule is checked every session, the screen and
reweight happen on the first trading day of each month. Alongside the equity curve it
records what it decided along the way — the full screen, the target book, and the daily
mark-to-market of every position — which is what the rest of this notebook plots.

In [ ]:
result = bt.run_backtest(raw_data, bench_df, compare_df, cfg)

equity = result["equity_curve"]          # date -> portfolio value, positions held
bench_curve = result["benchmark_curve"]  # buy & hold of cfg.compare_ticker
trades = result["trade_log"]             # every fill
screen = result["screen_log"]            # every symbol, every rebalance, every filter
rebalances = result["rebalance_log"]     # one summary row per rebalance date
allocations = result["allocation_log"]   # the target book set on each rebalance
holdings = result["holdings_value"]      # daily $ value per position, plus CASH

rebal_dates = list(rebalances["date"])

print(f"simulated {equity.index.min().date()} -> {equity.index.max().date()}  "
      f"({len(equity)} trading days, {len(rebal_dates)} rebalances)")
print(f"{len(trades)} fills, ${result['turnover_notional']:,.0f} traded notional")
print(f"final value ${equity['value'].iloc[-1]:,.0f} "
      f"from ${cfg.initial_capital:,.0f}")

---
# Part 1 — Stock selection

Every rebalance the whole universe is put through three pass/fail filters, and whatever
survives is ranked by relative strength:

| Stage | Rule |
|---|---|
| **Trend template** | Stage-2 uptrend: price above the mid and long SMAs, SMAs stacked, long SMA rising, inside the 52-week high/low band |
| **Tight base** | the recent price range is narrow enough to look like a base rather than a swing |
| **Liquidity** | minimum price and minimum average volume |
| **Relative strength** | weighted 3/6/9/12-month return *minus* the same for the benchmark; highest scores win the slots |

Pick the month you want to inspect here — the rest of Part 1 and the first chart of
Part 2 follow it.

In [ ]:
# Change the index to walk through the backtest month by month.
# rebal_dates[0] is the first rebalance, rebal_dates[-1] the most recent.
REBAL_DATE = rebal_dates[-1]

day = screen[screen["date"] == REBAL_DATE]
summary = rebalances.set_index("date").loc[REBAL_DATE]

print(f"Rebalance of {REBAL_DATE:%Y-%m-%d}: "
      f"{summary['n_qualified']:.0f} of {summary['n_universe']:.0f} names qualified, "
      f"top {summary['n_selected']:.0f} bought, "
      f"{summary['cash_weight']:.1%} left in cash")

picks = (day[day["selected"]]
         .sort_values("rs_rank")
         .set_index("rs_rank")[["symbol", "close", "rs_score", "pct_below_52w_high",
                                "pct_above_52w_low", "base_range_pct", "avg_volume"]])
picks.index.name = "rank"
display(picks.style.format({
    "close": "${:,.2f}", "rs_score": "{:+.3f}", "pct_below_52w_high": "{:.1f}%",
    "pct_above_52w_low": "{:.1f}%", "base_range_pct": "{:.1f}%", "avg_volume": "{:,.0f}",
}))

### The screen as a funnel

How many names each stage removes. A stage that never rejects anything is not doing any
work — the liquidity floor, for instance, is deliberately non-binding for QQQ members
and is there as a safety check.

In [ ]:
stages = [
    ("Universe", summary["n_universe"]),
    ("+ trend template", summary["n_pass_trend_template"]),
    ("+ tight base", summary["n_pass_tight_base"]),
    ("+ liquidity", summary["n_pass_liquidity"]),
    ("+ RS computable", summary["n_qualified"]),
    (f"Selected (top {cfg.max_positions})", summary["n_selected"]),
]
labels = [s for s, _ in stages]
counts = [float(c) for _, c in stages]

fig, ax = plt.subplots(figsize=(9, 3.6))
bars = ax.barh(range(len(counts)), counts,
               color=[STRAT] * (len(counts) - 1) + [ACCENT], height=0.65)
ax.set_yticks(range(len(counts)), labels)
ax.invert_yaxis()
ax.set_xlabel("names surviving")
ax.set_title(f"Screening funnel — {REBAL_DATE:%B %Y}")
ax.grid(axis="y", visible=False)
for bar, count, prev in zip(bars, counts, [counts[0]] + counts[:-1]):
    dropped = prev - count
    note = f"{count:.0f}" + (f"   (−{dropped:.0f})" if dropped > 0 else "")
    ax.text(bar.get_width() + max(counts) * 0.01, bar.get_y() + bar.get_height() / 2,
            note, va="center", fontsize=9)
ax.set_xlim(0, max(counts) * 1.15)
plt.tight_layout()
plt.show()

### Which rule rejected whom

The funnel above applies the filters in order, so a name dropped early hides whatever
else it also failed. This counts each rule independently across the rejected names.

In [ ]:
rejected = day[~day["qualified"]]
failures = pd.Series({
    "trend template": (~rejected["pass_trend_template"]).sum(),
    "tight base": (~rejected["pass_tight_base"]).sum(),
    "liquidity": (~rejected["pass_liquidity"]).sum(),
    "no RS (short history)": rejected["rs_score"].isna().sum(),
}).sort_values()

fig, ax = plt.subplots(figsize=(8, 2.8))
ax.barh(failures.index, failures.values, color=NEG, alpha=0.75, height=0.6)
ax.set_xlabel(f"names failing the rule (of {len(rejected)} rejected)")
ax.set_title(f"Reasons for rejection — {REBAL_DATE:%B %Y}")
ax.grid(axis="y", visible=False)
for y, v in enumerate(failures.values):
    ax.text(v + 0.4, y, f"{v}", va="center", fontsize=9)
plt.tight_layout()
plt.show()

### Relative-strength ranking

Every qualifier, sorted by RS score. The cut is purely mechanical: the top
`max_positions` get a slot, the rest wait for next month. A score of `+0.10` means the
name beat the benchmark by 10 percentage points on the weighted blend of lookbacks.

In [ ]:
qualified = day[day["qualified"]].sort_values("rs_score", ascending=False)

fig, ax = plt.subplots(figsize=(11, max(3, 0.24 * len(qualified))))
colors = [STRAT if s else CASH_C for s in qualified["selected"]]
ax.barh(qualified["symbol"], qualified["rs_score"] * 100, color=colors, height=0.7)
ax.invert_yaxis()
ax.axvline(0, color="black", lw=0.8)
if (~qualified["selected"]).any():
    cutoff = qualified[qualified["selected"]]["rs_score"].min() * 100
    ax.axvline(cutoff, color=ACCENT, ls="--", lw=1,
               label=f"cut-off (slot {cfg.max_positions})")
    ax.legend(loc="lower right")
ax.set_xlabel(f"RS score vs {cfg.benchmark_ticker} (percentage points)")
ax.set_title(f"Qualifiers ranked by relative strength — {REBAL_DATE:%B %Y}\n"
             f"filled = bought, grey = qualified but out-ranked")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

### What a selected name looks like

The trend template is a picture as much as a rule set. This draws it: price with the
three moving averages, the 52-week high/low band the filter checks against, and a marker
on every month the name was picked.

Change `SYMBOL` to any ticker in the universe — including one that was *never* picked,
to see why not.

In [ ]:
SYMBOL = picks["symbol"].iloc[0]   # the top-ranked pick this month

def plot_name(symbol, months_back=None):
    """Price, SMAs and the 52-week band, with selection and exit markers."""
    d = sig.add_indicators(raw_data[symbol], cfg)
    d = d[d.index >= equity.index.min()]
    if months_back:
        d = d[d.index >= d.index.max() - pd.DateOffset(months=months_back)]

    fig, (ax, axv) = plt.subplots(
        2, 1, figsize=(12, 6), sharex=True,
        gridspec_kw={"height_ratios": [3.2, 1], "hspace": 0.08})

    ax.fill_between(d.index, d["roll_52w_low"], d["roll_52w_high"],
                    color=STRAT, alpha=0.06, label="52-week range")
    ax.plot(d.index, d["close"], color="black", lw=1.2, label="close")
    ax.plot(d.index, d["sma_short"], color="#f59f00", lw=1, label=f"SMA{cfg.sma_short}")
    ax.plot(d.index, d["sma_mid"], color=STRAT, lw=1, label=f"SMA{cfg.sma_mid}")
    ax.plot(d.index, d["sma_long"], color=NEG, lw=1, label=f"SMA{cfg.sma_long}")

    chosen = screen[(screen["symbol"] == symbol) & screen["selected"]]["date"]
    chosen = [dt for dt in chosen if dt in d.index]
    if chosen:
        ax.scatter(chosen, d.loc[chosen, "close"], marker="^", s=70, zorder=5,
                   color=POS, label="selected at rebalance")

    exits = trades[(trades["symbol"] == symbol)
                   & (trades["action"] == "EXIT_TREND_BREAK")]
    exits = exits[exits["date"].isin(d.index)]
    if len(exits):
        ax.scatter(exits["date"], exits["price"], marker="v", s=70, zorder=5,
                   color=NEG, label=f"exit: close < SMA{cfg.sma_long}")

    ax.set_ylabel("price ($)")
    ax.set_title(f"{symbol} — trend template through the backtest window")
    ax.legend(ncol=3, fontsize=8, loc="upper left")

    axv.bar(d.index, d["volume"], color=CASH_C, width=1.0)
    axv.plot(d.index, d["avg_volume"], color=STRAT, lw=1,
             label=f"{cfg.avg_volume_lookback_days}d average")
    axv.set_ylabel("volume")
    axv.legend(fontsize=8)
    axv.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    plt.show()

plot_name(SYMBOL)

### Selection over time

Two views of the same thing. The line shows how many names clear the screen each month —
it collapses towards zero in corrections, which is the mechanism that moves the strategy
into cash. The grid below shows *who* was held, month by month.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(rebalances["date"], rebalances["n_qualified"], marker="o", ms=4,
        color=STRAT, label="qualified")
ax.plot(rebalances["date"], rebalances["n_selected"], marker="o", ms=4,
        color=ACCENT, label="selected (bought)")
ax.axhline(cfg.max_positions, color="black", ls=":", lw=1,
           label=f"max_positions = {cfg.max_positions}")
ax.fill_between(rebalances["date"], 0, rebalances["n_qualified"],
                color=STRAT, alpha=0.08)
ax.set_ylabel("names")
ax.set_title("Names clearing the screen at each rebalance")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
grid = (screen[screen["selected"]]
        .assign(held=1)
        .pivot_table(index="symbol", columns="date", values="held", fill_value=0))
grid = grid.loc[grid.sum(axis=1).sort_values(ascending=False).index]
if len(grid) > 35:                       # keep the chart readable on a wide universe
    grid = grid.head(35)

fig, ax = plt.subplots(figsize=(12, max(4, 0.26 * len(grid))))
ax.pcolormesh(np.arange(grid.shape[1] + 1), np.arange(grid.shape[0] + 1),
              grid.values, cmap="Blues", vmin=0, vmax=1.4, edgecolors="white", lw=0.5)
ax.set_yticks(np.arange(len(grid)) + 0.5, grid.index, fontsize=8)
tick_pos = np.arange(grid.shape[1]) + 0.5
step = max(1, len(tick_pos) // 14)
ax.set_xticks(tick_pos[::step],
              [d.strftime("%b %y") for d in grid.columns[::step]], fontsize=8)
ax.invert_yaxis()
ax.grid(False)
ax.set_title(f"Who was held, rebalance by rebalance "
             f"(most-selected {len(grid)} names)")
plt.tight_layout()
plt.show()

held_counts = grid.sum(axis=1).sort_values(ascending=False)
print(f"{screen['selected'].sum()} slot-months filled by "
      f"{screen[screen['selected']]['symbol'].nunique()} distinct names")
print("most persistent holdings:",
      ", ".join(f"{s} ({int(n)}x)" for s, n in held_counts.head(8).items()))

---
# Part 2 — Portfolio construction

The ranking from Part 1 turns into an actual book here. This strategy does **not** run a
mean-variance optimiser: the sizing rule *is* the optimisation, and it is deliberately
simple.

- `position_sizing = "fixed_slot"` — every name gets `1 / max_positions` of portfolio
  value, so a thin month leaves the balance in cash instead of concentrating into the
  two names that happened to qualify.
- `position_sizing = "equal_weight_of_qualifiers"` — the alternative: split 100% of
  capital across however many qualify, however few.
- `rebalance_mode = "full_equal_weight"` — the whole book is liquidated and rebuilt each
  month, which is the simplest way to hit exact weights (a live implementation would
  trade only the deltas and pay far less in costs).

Both knobs live in the `[portfolio]` table of `config.toml`.

In [ ]:
alloc = allocations[allocations["date"] == REBAL_DATE].sort_values("rs_rank")
cash_weight = float(summary["cash_weight"])

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(alloc["symbol"], alloc["weight"] * 100, color=STRAT, label="positions")
if cash_weight > 1e-9:
    ax.bar(["CASH"], [cash_weight * 100], color=CASH_C, label="unallocated cash")
ax.axhline(100 / cfg.max_positions, color=ACCENT, ls="--", lw=1,
           label=f"slot size = 100 / {cfg.max_positions} = {100 / cfg.max_positions:.1f}%")
ax.set_ylabel("% of portfolio value")
ax.set_title(f"Target book set on {REBAL_DATE:%d %B %Y}  —  "
             f"${summary['capital']:,.0f} to allocate\n"
             f"flat by construction; a month with fewer qualifiers leaves a cash bar")
ax.tick_params(axis="x", rotation=90, labelsize=8)
ax.set_ylim(0, max(alloc["weight"].max(), cash_weight) * 100 * 1.35)
ax.legend(fontsize=8, loc="upper right")
ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

display(alloc.set_index("rs_rank")[["symbol", "price", "shares", "target_value", "weight"]]
        .style.format({"price": "${:,.2f}", "shares": "{:,.2f}",
                       "target_value": "${:,.0f}", "weight": "{:.2%}"}))

### Composition drift between rebalances

Weights are equal only on the day they are set. Between rebalances the winners grow and
the losers shrink, and any name stopped out by the daily exit rule turns into cash until
the next rebalance — that grey band is the strategy sitting out.

In [ ]:
weights = holdings.div(holdings.sum(axis=1), axis=0)

# Stack the biggest average positions individually and pool the rest, otherwise the
# legend is 40 names long and the chart says nothing.
avg = weights.drop(columns="CASH").mean().sort_values(ascending=False)
top = list(avg.head(10).index)
stack = pd.DataFrame({
    **{s: weights[s] for s in top},
    "other positions": weights.drop(columns=["CASH"] + top).sum(axis=1),
    "cash": weights["CASH"],
})

fig, ax = plt.subplots(figsize=(12, 5))
colors = list(plt.cm.tab20(np.linspace(0, 1, len(top)))) + ["#f1f3f5", CASH_C]
ax.stackplot(stack.index, stack.T.values * 100, labels=stack.columns, colors=colors)
ax.set_ylim(0, 100)
ax.set_ylabel("% of portfolio value")
ax.set_title("Portfolio composition, day by day")
ax.legend(ncol=4, fontsize=7.5, loc="lower center", bbox_to_anchor=(0.5, -0.42))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
ax.plot(equity.index, equity["n_positions"], color=STRAT, lw=1.2)
ax.set_ylabel("positions held", color=STRAT)
ax.set_ylim(0, cfg.max_positions + 1)
ax.axhline(cfg.max_positions, color="black", ls=":", lw=1)

ax2 = ax.twinx()
ax2.fill_between(weights.index, 0, weights["CASH"] * 100, color=CASH_C, alpha=0.8)
ax2.set_ylabel("cash weight (%)", color="#495057")
# Scaled to the cash actually held -- read the right-hand axis, not the area.
ax2.set_ylim(0, min(100, max(20, weights["CASH"].max() * 100 * 1.3)))
ax2.grid(False)
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.set_title("Exposure: how many slots were filled, and how much sat in cash")
plt.tight_layout()
plt.show()

print(f"average positions held: {equity['n_positions'].mean():.1f} "
      f"of {cfg.max_positions}")
print(f"average cash weight: {weights['CASH'].mean():.1%}   "
      f"(fully invested on {(weights['CASH'] < 0.01).mean():.0%} of days)")

### Turnover

Full liquidation and rebuild every month is expensive, and this is the bill. Each bar is
the notional traded on that rebalance, split into the sells that fund it and the buys
that follow. Costs are already inside the equity curve at `txn_cost_bps` per leg.

In [ ]:
# Bucketed by month, not by fill date: the monthly rebalance and any stop-outs
# that happened in between belong to the same month's trading bill.
monthly_turnover = (
    trades.assign(notional=trades["shares"] * trades["price"],
                  month=trades["date"].dt.to_period("M").dt.to_timestamp())
    .pivot_table(index="month", columns="action", values="notional",
                 aggfunc="sum", fill_value=0.0))
for col in ("SELL_REBAL", "EXIT_TREND_BREAK", "BUY_REBAL"):
    if col not in monthly_turnover:
        monthly_turnover[col] = 0.0

fig, ax = plt.subplots(figsize=(11, 4))
layers = [("SELL_REBAL", NEG, "rebalance sells"),
          ("EXIT_TREND_BREAK", ACCENT, "stopped out (below SMA200)"),
          ("BUY_REBAL", POS, "rebalance buys")]
bottom = np.zeros(len(monthly_turnover))
for col, color, label in layers:
    ax.bar(monthly_turnover.index, monthly_turnover[col], width=22, bottom=bottom,
           color=color, alpha=0.85, label=label)
    bottom += monthly_turnover[col].values
ax.set_ylabel("notional traded ($)")
ax.set_title("Turnover per month")
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(lambda v, _: f"${v:,.0f}")
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.tight_layout()
plt.show()

years = (equity.index[-1] - equity.index[0]).days / 365.25
est_cost = result["turnover_notional"] * cfg.txn_cost_bps / 10000
print(f"total traded notional: ${result['turnover_notional']:,.0f} "
      f"({result['turnover_notional'] / cfg.initial_capital / years:.1f}x "
      f"initial capital per year)")
print(f"estimated transaction costs: ${est_cost:,.0f} "
      f"({est_cost / cfg.initial_capital:.1%} of starting capital)")

---
# Part 3 — Profit and loss

What the book above actually earned, against simply holding the index.

In [ ]:
fig, (ax, axd) = plt.subplots(
    2, 1, figsize=(12, 6.5), sharex=True,
    gridspec_kw={"height_ratios": [2.6, 1], "hspace": 0.1})

ax.plot(equity.index, equity["value"], color=STRAT, lw=1.6, label="Momentum strategy")
ax.plot(bench_curve.index, bench_curve.values, color=BENCH, lw=1.4, ls="--",
        label=f"{cfg.compare_ticker} buy & hold")
ax.axhline(cfg.initial_capital, color="black", lw=0.8, ls=":")
ax.set_ylabel("portfolio value ($)")
ax.set_title("Strategy vs. buy & hold")
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(lambda v, _: f"${v:,.0f}")

for series, color, label in ((equity["value"], STRAT, "strategy"),
                             (bench_curve, BENCH, cfg.compare_ticker)):
    dd = (series / series.cummax() - 1) * 100
    axd.fill_between(dd.index, dd.values, 0, color=color, alpha=0.35, label=label)
axd.set_ylabel("drawdown (%)")
axd.legend(fontsize=8, loc="lower left")
axd.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
plt.show()

In [ ]:
bench_frame = pd.DataFrame({"value": bench_curve, "n_positions": 1})
report = pd.DataFrame({
    "strategy": bt.performance_report(equity, cfg),
    f"{cfg.compare_ticker} buy & hold": bt.performance_report(bench_frame, cfg),
})
report.loc["final_value"] = [equity["value"].iloc[-1], bench_curve.iloc[-1]]
report.loc["profit_usd"] = report.loc["final_value"] - cfg.initial_capital

display(report.style.format("{:,.2f}"))

edge = (report.loc["total_return_pct", "strategy"]
        - report.loc["total_return_pct", f"{cfg.compare_ticker} buy & hold"])
print(f"strategy {'beat' if edge > 0 else 'trailed'} {cfg.compare_ticker} "
      f"by {abs(edge):.1f} percentage points over "
      f"{(equity.index[-1] - equity.index[0]).days / 365.25:.1f} years")

### Monthly returns

Where the total came from. Rows are years, columns months; the strategy's month is
green when it made money and red when it lost.

In [ ]:
monthly = equity["value"].resample("ME").last().pct_change().dropna() * 100
table = (monthly.to_frame("ret")
         .assign(year=lambda d: d.index.year, month=lambda d: d.index.month)
         .pivot_table(index="year", columns="month", values="ret"))
table.columns = [pd.Timestamp(2000, m, 1).strftime("%b") for m in table.columns]

limit = np.nanmax(np.abs(table.values))
fig, ax = plt.subplots(figsize=(11, 0.55 * len(table) + 1.6))
ax.pcolormesh(np.arange(table.shape[1] + 1), np.arange(table.shape[0] + 1),
              np.ma.masked_invalid(table.values), cmap="RdYlGn",
              vmin=-limit, vmax=limit, edgecolors="white", lw=1.5)
for r in range(table.shape[0]):
    for c in range(table.shape[1]):
        v = table.values[r, c]
        if not np.isnan(v):
            ax.text(c + 0.5, r + 0.5, f"{v:.1f}", ha="center", va="center", fontsize=8.5)
ax.set_xticks(np.arange(table.shape[1]) + 0.5, table.columns)
ax.set_yticks(np.arange(table.shape[0]) + 0.5, table.index)
ax.invert_yaxis()
ax.grid(False)
ax.set_title("Monthly return (%)")
plt.tight_layout()
plt.show()

print(f"best month {monthly.max():+.1f}%   worst month {monthly.min():+.1f}%   "
      f"positive months {(monthly > 0).mean():.0%}")

### Trade-level P&L

Fills are matched into round trips (first in, first out) so each closed position gets a
P&L, net of the transaction cost on both legs. Because the whole book is rebuilt monthly,
most round trips are one month long — a name that keeps ranking is sold and re-bought
rather than held, so its long ride shows up as a chain of monthly trades.

In [ ]:
def build_round_trips(trade_log, cfg):
    """Match sells against earlier buys FIFO and cost each closed lot."""
    fee_rate = cfg.txn_cost_bps / 10000.0
    open_lots: dict[str, list[list]] = {}
    closed = []
    for t in trade_log.itertuples():        # already in chronological order
        if t.action == "BUY_REBAL":
            open_lots.setdefault(t.symbol, []).append([t.date, t.shares, t.price])
            continue
        remaining, lots = t.shares, open_lots.get(t.symbol, [])
        while remaining > 1e-9 and lots:
            entry_date, lot_shares, entry_price = lots[0]
            qty = min(remaining, lot_shares)
            fees = qty * (entry_price + t.price) * fee_rate
            closed.append({
                "symbol": t.symbol, "entry_date": entry_date, "exit_date": t.date,
                "shares": qty, "entry_price": entry_price, "exit_price": t.price,
                "exit_reason": t.action,
                "pnl": qty * (t.price - entry_price) - fees,
                "return_pct": (t.price / entry_price - 1) * 100,
                "held_days": (t.date - entry_date).days,
            })
            lots[0][1] -= qty
            remaining -= qty
            if lots[0][1] <= 1e-9:
                lots.pop(0)
    return pd.DataFrame(closed)

round_trips = build_round_trips(trades, cfg)

wins = round_trips[round_trips["pnl"] > 0]
losses = round_trips[round_trips["pnl"] <= 0]
stats = pd.Series({
    "closed round trips": len(round_trips),
    "win rate": f"{len(wins) / len(round_trips):.1%}",
    "average win": f"${wins['pnl'].mean():,.0f}  ({wins['return_pct'].mean():+.1f}%)",
    "average loss": f"${losses['pnl'].mean():,.0f}  ({losses['return_pct'].mean():+.1f}%)",
    "profit factor": f"{wins['pnl'].sum() / abs(losses['pnl'].sum()):.2f}",
    "realised P&L": f"${round_trips['pnl'].sum():,.0f}",
    "median holding period": f"{round_trips['held_days'].median():.0f} days",
}, name="")
display(stats.to_frame())

display(round_trips.sort_values("pnl", ascending=False)
        .head(10)[["symbol", "entry_date", "exit_date", "held_days",
                   "return_pct", "pnl", "exit_reason"]]
        .style.format({"return_pct": "{:+.1f}%", "pnl": "${:,.0f}"})
        .hide(axis="index"))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(round_trips["return_pct"], bins=40, color=STRAT, alpha=0.85)
ax1.axvline(0, color="black", lw=1)
ax1.axvline(round_trips["return_pct"].mean(), color=ACCENT, ls="--", lw=1.2,
            label=f"mean {round_trips['return_pct'].mean():+.1f}%")
ax1.set_xlabel("return per round trip (%)")
ax1.set_ylabel("count")
ax1.set_title("Distribution of trade returns")
ax1.legend(fontsize=8)

by_reason = round_trips.groupby("exit_reason")["pnl"].agg(["count", "sum", "mean"])
ax2.bar(by_reason.index, by_reason["sum"],
        color=[POS if v > 0 else NEG for v in by_reason["sum"]], alpha=0.85)
ax2.axhline(0, color="black", lw=1)
ax2.set_ylabel("realised P&L ($)")
ax2.set_title("P&L by exit reason")
ax2.yaxis.set_major_formatter(lambda v, _: f"${v:,.0f}")
span = by_reason["sum"].abs().max()
for x, (n, total) in enumerate(zip(by_reason["count"], by_reason["sum"])):
    ax2.text(x, total - np.sign(total) * span * 0.06, f"{n} trades",
             ha="center", va="center", fontsize=8)
ax2.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()

display(by_reason.style.format({"sum": "${:,.0f}", "mean": "${:,.0f}"}))

### Who made the money

Realised P&L per name, summed across every round trip it went through. A short tail on
either side means the result rests on a handful of names — worth knowing before trusting
the headline return.

In [ ]:
by_symbol = round_trips.groupby("symbol")["pnl"].sum().sort_values()
tails = pd.concat([by_symbol.head(12), by_symbol.tail(12)])
tails = tails[~tails.index.duplicated()]

fig, ax = plt.subplots(figsize=(10, max(4, 0.3 * len(tails))))
ax.barh(tails.index, tails.values,
        color=[POS if v > 0 else NEG for v in tails.values], alpha=0.85)
ax.axvline(0, color="black", lw=0.8)
ax.set_xlabel("realised P&L ($)")
ax.set_title("Biggest contributors and detractors")
ax.grid(axis="y", visible=False)
plt.tight_layout()
plt.show()

top5 = by_symbol.tail(5)
print(f"top 5 names contributed ${top5.sum():,.0f} of "
      f"${round_trips['pnl'].sum():,.0f} realised "
      f"({top5.sum() / round_trips['pnl'].sum():.0%})")

---
## Read this before you believe any of the above

- **Survivorship bias.** The universe is *today's* QQQ membership applied backwards
  (`qqq_universe.py`). Names that were dropped from the index during the window —
  because they collapsed, were acquired or were demoted — are simply absent, which
  flatters the result. This is the single largest distortion in the backtest.
- **Costs are a flat assumption.** `txn_cost_bps` covers spread and slippage as one
  number per leg. There is no market impact, no borrow, no tax.
- **Full monthly rebuild.** `full_equal_weight` sells and re-buys everything each month.
  A live implementation would trade the deltas only; the turnover chart in Part 2 shows
  what that simplification costs.
- **Mechanised judgment.** "Properly formed base" is a range-tightness proxy. The
  original methodology reads the chart; this reads one number.

To take it further: change a threshold in `config.toml`, re-run from *Section 3* down,
and compare. `python main.py` runs the same backtest headless and writes
`equity_curve.csv`, `trade_log.csv` and `backtest_result.png`.